# 04 — Generalize, don't memorize

*Module 00 · Getting Started — notebook 4 of 11*

We make the first real move of modelling here: setting data aside. It is the discipline that separates *measuring* learning from fooling ourselves.

**Prerequisites:** 01 (what is ML), 02 (the feature space), 03 (looking at the data).

**What you'll be able to do**
- Explain the **cardinal sin** — scoring a model on its own training data — and why a perfect training score proves nothing.
- Make a **stratified train/test split** and check it kept the class balance.
- State the **i.i.d. assumption** as a choice, and recognize **data leakage**.
- Describe the honest loop: *fit → predict → evaluate on held-out data*.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

from ml_course import colors, datasets, viz

np.random.seed(0)
viz.use_course_style()

df = datasets.load_penguins()
X, y = datasets.penguins_xy()
FEATURES = datasets.PENGUINS_FEATURES
SPECIES_ORDER = ["Adelie", "Gentoo"]

## Where we are

Notebook 03 looked closely at the data — its balance, shapes, and scales — without building anything. Now we take the step every honest workflow shares before training a model: we **set some data aside**, and promise not to look at it until the very end.

Why such a strict rule? Because of a trap that catches everyone at least once.

## The cardinal sin

Imagine a student handed the exam with the answer key stapled to it. They score 100% — and have shown nothing about whether they learned the material. Hand them a fresh exam, and you find out.

A model is no different. If we train it on some penguins and then **score it on those same penguins**, a model that merely memorized them will look perfect. That number measures memorization, not learning. Scoring a model on its training data is the cardinal sin of machine learning — and it is surprisingly easy to commit by accident.

## Memorize vs generalize

What we actually want is a model that does well on **new** penguins — ones it has never seen. That ability is called **generalization**, and it is the whole point. We can estimate it honestly only on data the model was not trained on.

So we split our penguins into two parts: a **training set** the model learns from, and a **test set** we keep sealed and use once, at the end, to estimate how well the model generalizes.

## The train/test split

We hold out a slice of the data — here 25% — as the test set, and train on the remaining 75%. One care to take: **stratify** the split, so the training and test sets keep the same class proportions as the full dataset. Without stratification, a random split can land too few of a rare class on one side. We also fix the random seed, so the split is reproducible.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=0, stratify=y
)

print(f"training set: {len(X_train)} penguins")
print(f"test set    : {len(X_test)} penguins")
print("\ntraining class proportions:")
print(y_train.value_counts(normalize=True).round(3).to_string())
print("\ntest class proportions:")
print(y_test.value_counts(normalize=True).round(3).to_string())

### Read the output

The split gives 205 training penguins and 69 test penguins. Thanks to stratification, both sets carry the same balance as the whole dataset — about 55.1% Adélie and 44.9% Gentoo — so the test set is a fair miniature of the problem, not skewed by the luck of the draw. That fairness matters most when a class is rare: an unstratified split could leave almost none of it in the test set, and then the test score would say little about that class.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5.5))
ax.scatter(
    X_train[FEATURES[0]],
    X_train[FEATURES[1]],
    color=colors.COLORS["train"],
    edgecolor=colors.COLORS["text"],
    linewidth=0.4,
    s=40,
    label="training set",
    alpha=0.8,
)
ax.scatter(
    X_test[FEATURES[0]],
    X_test[FEATURES[1]],
    color=colors.COLORS["test"],
    edgecolor=colors.COLORS["text"],
    linewidth=0.4,
    s=55,
    marker="^",
    label="test set (sealed)",
    alpha=0.9,
)
ax.set_xlabel(FEATURES[0])
ax.set_ylabel(FEATURES[1])
ax.set_title("Training set vs held-out test set")
ax.legend()
plt.show()

### Read the figure

The two colours are the same penguins, split by *role* rather than by species: training points in one colour, the held-out test points (triangles) in another. The test points are scattered through the same regions as the training points — that is what a stratified random split gives us — but from now on we treat them as if they did not exist, until the single moment at the end when we score the model. That sealed-envelope discipline is what makes a test score worth believing.

## A rote memorizer

To *feel* the cardinal sin, meet a deliberately silly "model." It does one thing: it **memorizes** every training penguin's measurements and species. Shown a new penguin, it looks for those exact measurements in its memory; if it finds them, it returns the stored species; if not, it shrugs and guesses the most common species. This is not a method we would ever use — it is a cautionary tale with code.

In [ ]:
# Store each training penguin's answer, keyed by its exact (bill, flipper) measurements.
seen = {
    (bill, flipper): species
    for bill, flipper, species in zip(X_train[FEATURES[0]], X_train[FEATURES[1]], y_train)
}
majority_species = y_train.mode().iloc[0]


def memorizer_predict(features):
    """Return the stored species for exact-match measurements, else the majority species."""
    return np.array(
        [
            seen.get((bill, flipper), majority_species)
            for bill, flipper in zip(features[FEATURES[0]], features[FEATURES[1]])
        ]
    )


train_right = (memorizer_predict(X_train) == y_train.to_numpy()).mean()
test_right = (memorizer_predict(X_test) == y_test.to_numpy()).mean()
majority_share = y_test.value_counts(normalize=True).max()

print(f"fraction right on TRAIN: {train_right:.3f}")
print(f"fraction right on TEST : {test_right:.3f}")
print(f"(always guessing '{majority_species}' scores {majority_share:.3f} on the test set)")

### Read the output

On the training set the memorizer is perfect — 100% — because it has every training penguin memorized. On the held-out test set it gets about 55% right: no better than the **baseline**, a model that ignores the measurements and always guesses the common species (Adélie). A few test penguins do happen to repeat a training penguin's exact measurements, and the memorizer recognizes those — but keying on exact measurements buys nothing on genuinely new penguins, so it falls back to that majority guess almost everywhere. (At this particular split the two scores land on the same number; the durable point is the collapse to the baseline, not the exact digits.)

That gap is the entire lesson of this notebook. A perfect training score told us nothing about new penguins; only the held-out score revealed that the memorizer learned nothing that generalizes. (We have been calling it "the fraction it gets right" — notebook 06 makes this a proper metric, **accuracy**, and shows where even that can mislead.)

## The assumption underneath

The split rests on a quiet assumption: that the training and test penguins are drawn from the **same population** — independent draws, identically distributed (often shortened to **i.i.d.**). That is what lets a held-out score stand in for performance on future penguins.

We *adopt* that assumption here; it is a choice, not a guarantee. Real penguins were measured across different islands and years, and such structure can break i.i.d. — a future penguin might come from a colony the training set barely covered. We name this now so it is not a surprise later.

## Leakage

The cardinal sin is the simplest example of **data leakage**: letting information from the test set sneak into how we build or report the model. The fix we applied above — sealing the test set and scoring it once, at the end — prevents the obvious leak.

Subtler leaks exist. Preparing the data — for instance, rescaling features using the test set's values *before* splitting — is a leak too, because the model then indirectly "sees" the test data. We meet that one, and how to avoid it, in notebook 11.

## The honest loop

Every method in this course follows the same honest cycle:

1. **fit** — let the model learn its rule from the *training* set;
2. **predict** — apply that rule to new examples;
3. **evaluate** — measure quality on the *held-out* test set.

Notebook 05 builds our first real classifier — the nearest centroid — and runs exactly this loop, end to end.

## The words we'll keep using

Added this notebook:

- **Cardinal sin** — scoring a model on its own training data; it measures memorization, not learning.
- **Train/test split** — dividing data into a part to learn from and a part held back to judge on.
- **Test (held-out) set** — data sealed away and used once, at the end, to estimate generalization.
- **Generalization** — doing well on new, unseen examples.
- **Stratification** — splitting so each part keeps the same class proportions.
- **Leakage** — letting test information influence training or reporting.
- **i.i.d.** — independent and identically distributed: the assumption that train and test come from the same population.
- **Baseline** — a trivial reference score to beat (here, always guessing the common class). More in notebook 06.

## Your turn

1. A friend says their model gets 100% accuracy. Before being impressed, what is the first question you ask?
2. You choose the test fraction. What do you trade off by holding out 10% of the data versus 50%?
3. Name one way information from the test set could leak into training. And: why is stratification especially important when one class is rare?

Write your answers in a new cell. Notebook 05 puts this loop to work with a real classifier.

## What you built

You've taken the first methodological step, and it is one of the most important in the whole course:

- You can name the **cardinal sin** — and you have *felt* it, in the memorizer's 100% training score that collapsed to no-better-than-the-baseline on new data.
- You can make a **stratified train/test split** and check it kept the class balance.
- You can state the **i.i.d. assumption** as a choice, and you can recognize **leakage**.
- You can describe the honest loop, **fit → predict → evaluate**, that every method ahead will follow.

With the data split and the rules of the game clear, we are ready to build something real.

## References

- G. James, D. Witten, T. Hastie, R. Tibshirani, *An Introduction to Statistical Learning*, 2nd ed., Springer, 2021 — §2.2 (assessing model accuracy) and chapter 5 (resampling methods). DOI: 10.1007/978-1-0716-1418-1
- scikit-learn developers, *Common pitfalls and recommended practices* — data leakage and held-out evaluation. https://scikit-learn.org/stable/common_pitfalls.html

---
Previous: **03 — Look before you model** · Next: **05 — Your first classifier: nearest centroid**